# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/oumaklaus/ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

My lane (chosen last week) is Refresh / Content Opportunity Scoring. I think this is best treated as a **ranking / scoring** task, with a classifier doing the work underneath.

My reasoning: the decision is "which pages should an editor review first", and that is an ordering question, not a plain yes/no. So what I want to hand the editor is a ranked queue where every page gets a priority score, and they work down the list until they run out of time that week. That is scoring, and judging the top of an ordered list is a ranking problem.

Underneath the score, I plan to train a binary classifier that estimates how likely a page is to be a refresh candidate, then use that probability as the score. So it is really "a classifier used for ranking". It is not clustering, because I do have a specific thing I care about rather than just "what groups exist", and it is not plain classification for its own sake, because a hard yes/no would throw away the ordering the editor actually needs.

In [7]:
# Load the lane's slice of the starter data once, for the whole notebook.
import numpy as np
import pandas as pd

#df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df = pd.read_csv("https://github.com/oumaklaus/ml-internship/raw/main/data/raw/content_refresh_anonymized.csv")
print(f"rows (pages): {len(df):,}   columns: {df.shape[1]}   clients: {df['client_id'].nunique()}")

# The lane keeps pages that are old enough to judge and actually visible in search
# (mirrors the starter pipeline's filters: content_age_days >= 90 and impressions_90d > 0).
lane = df[(df["content_age_days"] >= 90) & (df["impressions_90d"] > 0)].copy()
print(f"lane slice after filters (age>=90 & impressions>0): {len(lane):,} pages")

rows (pages): 30,000   columns: 44   clients: 32
lane slice after filters (age>=90 & impressions>0): 30,000 pages


## 2. Target or proxy

What I would predict, per page, is the chance that it is a refresh candidate: a page whose search performance is slipping while it still pulls demand.

The label I can build from the starter data is a **proxy**, not a clean observed outcome. The starter pipeline uses:

```text
is_declining_label = (trend_direction == "down")
```

`trend_direction` is a defined rule, not a future measurement. It is a bucket worked out from current-window inputs (last-30-day vs previous-30-day impressions). The framing skill is blunt about this: the target should be observed, not defined, because a label that comes from a rule can just teach the model the rule instead of the world. So I treat `is_declining_label` honestly as a beginner proxy. It is fine for showing the whole workflow on the small starter slice, but it is not what I would call a strong final target.

Two things I am committing to now because of that:

1. Since the label is built from `trend_direction` (which is built from `trend_pct`), neither of those columns can ever be a feature. That is the exact leakage the starter notebook 02 shows.
2. A stronger, genuinely observed target would look forward in time: build features from an earlier window and measure decline in a later window (for example, features from the prior 90 days, then whether impressions drop over the next 30). I would need the bigger warehouse daily data for that, so I will pin it down when I write the data contract.

In [2]:
# Build the proxy target and sketch what the target column looks like.
lane["is_declining_label"] = (lane["trend_direction"] == "down").astype(int)

base_rate = lane["is_declining_label"].mean()
print(f"proxy positive rate (is_declining_label==1): {base_rate:.3f}  "
      f"({lane['is_declining_label'].sum():,} of {len(lane):,})")
print("\nlabel is well-balanced, so ROC-AUC and precision@K are both meaningful:")
print(lane["is_declining_label"].value_counts().rename({0: "not declining", 1: "declining"}).to_string())

# Quarantine list: columns that DERIVE the label and must never be features.
LEAKAGE_COLS = ["trend_direction", "trend_pct"]
print(f"\ncolumns quarantined from features (derive the label): {LEAKAGE_COLS}")

proxy positive rate (is_declining_label==1): 0.542  (16,262 of 30,000)

label is well-balanced, so ROC-AUC and precision@K are both meaningful:
is_declining_label
declining        16262
not declining    13738

columns quarantined from features (derive the label): ['trend_direction', 'trend_pct']


## 3. Success metric

My main metric is **precision@K**. I will report precision@50, and also @20.

Why this one, and why I can defend it: the editor never works the whole list. They review the top of the queue until their capacity runs out. So "good" means the pages I put at the top really are the ones worth fixing. Precision@50 says exactly that, of the 50 pages ranked first, how many are true refresh candidates. Plain accuracy would be the wrong choice here. With a positive rate near 54%, a model can look accurate while still ordering the top of the list badly, and the top is the only part the editor ever sees.

What counts as good: beat the base rate (about 0.54, which is the precision you would get by grabbing 50 pages at random), and then beat a simple, readable rule baseline that I will build before the model. The starter reference makes the point that the bar is real, its plain rule baseline scored 0.240 at precision@50 while a random forest reached 0.740 on that slice. I do not get to inherit those numbers though. I have to earn my own on my slice, with clients held out of training.

I will also watch ROC-AUC as a secondary sanity check on the overall ranking quality (the label is balanced, so AUC is still informative). Precision@K is what decides; AUC is just context.

In [3]:
# Show precision@K is computable TODAY, on a trivial baseline, so the metric is real.
def precision_at_k(scores, y, k):
    order = np.argsort(-np.asarray(scores, dtype=float))
    return float(np.asarray(y)[order[:k]].mean())

y = lane["is_declining_label"]

# A throwaway single-signal "baseline": rank by low CTR (a plausible refresh hint).
naive_score = -lane["ctr"].fillna(lane["ctr"].max())
for k in (20, 50):
    print(f"precision@{k:<2d} ranking by low-CTR: {precision_at_k(naive_score, y, k):.3f}")
print(f"base rate (random pick):        {y.mean():.3f}")
print("\n-> the metric runs today; next step later is a proper rule baseline to beat.")

precision@20 ranking by low-CTR: 0.450
precision@50 ranking by low-CTR: 0.500
base rate (random pick):        0.542

-> the metric runs today; next step later is a proper rule baseline to beat.


## 4. The unit of analysis, as a real dataframe

**One row = one page** (one pseudonymous `content_id`), summarised over the trailing-90-day window. Not a client, not a day, not a query. The output ranks these rows; the editor acts on one page at a time.

The cell below shows the grain literally: a handful of rows with the columns a reviewer would actually reason about, plus a check that `content_id` is unique (one row per page, no accidental duplication).

In [4]:
# Prove the grain: one row per page, and no duplicate content_ids.
dupes = lane["content_id"].duplicated().sum()
print(f"rows: {len(lane):,}   unique content_id: {lane['content_id'].nunique():,}   duplicate ids: {dupes}")
assert dupes == 0, "grain broken: content_id should be unique (one row per page)"

# What one row of the unit of analysis looks like (reviewer-facing columns only; no raw/private fields).
show_cols = ["content_id", "impressions_90d", "clicks_90d", "avg_position",
             "ctr", "days_since_last_update", "content_age_days",
             "word_count", "is_declining_label"]
lane[show_cols].head(5)

rows: 30,000   unique content_id: 30,000   duplicate ids: 0


,content_id,impressions_90d,clicks_90d,avg_position,ctr,days_since_last_update,content_age_days,word_count,is_declining_label
0,content_304f48230142,3803,29,10.6,0.76,20,187,3221.0,1
1,content_a1fb4e703a9e,15320,7,20.3,0.05,25,445,2481.0,1
2,content_9aa793d4d895,12581,11,36.5,0.09,20,141,3515.0,1
3,content_331d6c4de07b,11751,58,6.2,0.49,22,463,NaN,0
4,content_d99b7a2d90ca,19140,24,44.0,0.13,14,263,2803.0,1


## 5. Why ML beats a fixed rule here

A fixed rule (something like "flag any page not updated in 180+ days") falls short here for two reasons I can actually measure, both shown in the code below.

First, no single signal orders the queue on its own. The strongest single feature only correlates about 0.16 with the proxy label, and ranking by any one signal barely beats the 54% base rate. Some single-signal rankings even come in under it. The useful information lives in how demand, position, freshness, CTR and engagement move together, not in any one column by itself. That "lots of signals, all tangled" situation is where a learned model earns its place over an if-statement.

Second, a rule flags a bucket but cannot rank inside it. A threshold labels a whole group "stale" and stops there, giving every page in that group the same verdict, so it can't tell the editor which of them to open first. A scoring model gives a continuous priority instead, so the top of the list is genuinely ordered.

I want to be careful not to oversell it though. The model gives a decision-support ranking, a prioritised shortlist for a human to review. It is not an automated verdict, and it is not proof that refreshing a page will bring traffic back. The plain rule baseline still matters as the honest bar the model has to clear, and if the model can't beat it, that is a result worth reporting rather than a failure to hide.

The content action this supports: the editor opens the ranked queue, and for each page near the top decides to refresh, rewrite or expand it, protect it, or leave it, guided by short reason codes attached to each page (things like `declining_with_demand` or `low_ctr_visible_page`).

In [5]:
# Evidence for claim 1: single signals are weak and don't cleanly order the queue.
def precision_at_50(scores, y):
    order = np.argsort(-np.asarray(scores, dtype=float))
    return float(np.asarray(y)[order[:50]].mean())

signals = {
    "days_since_last_update": lane["days_since_last_update"],
    "impressions_90d":        lane["impressions_90d"],
    "low ctr first":          -lane["ctr"].fillna(lane["ctr"].max()),
    "worse position first":   -lane["avg_position"].replace(0, np.nan).fillna(lane["avg_position"].max()),
}
print(f"base rate (random top-50): {y.mean():.3f}")
print("precision@50 for each SINGLE-signal ranking:")
for name, s in signals.items():
    print(f"  {name:24s}: {precision_at_50(s, y):.3f}")

# strongest single-signal correlation with the label -> weak, so combine signals (ML).
num = ["days_since_last_update", "avg_position", "impressions_90d", "ctr",
       "engagement_rate", "word_count", "content_age_days", "search_volume"]
corr = lane[num + ["is_declining_label"]].corr(numeric_only=True)["is_declining_label"].drop("is_declining_label")
print(f"\nstrongest single-signal |correlation| with the proxy label: {corr.abs().max():.3f}")
print("-> no single rule dominates; the signal is in the COMBINATION. That is the case for ML.")

base rate (random top-50): 0.542
precision@50 for each SINGLE-signal ranking:
  days_since_last_update  : 0.540
  impressions_90d         : 0.420
  low ctr first           : 0.500
  worse position first    : 0.160

strongest single-signal |correlation| with the proxy label: 0.164
-> no single rule dominates; the signal is in the COMBINATION. That is the case for ML.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled: markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/`, then submit your repo URL on the card. Done.